# Part 2 - SAE + Concept Naming + Full Reliability Evaluation



## 0. Mount Drive and define paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/xai-project-5-main")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs_part2_full_eval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDINGS_PATH = DATA_DIR / "image_embeddings_3000_clip.npy"
METADATA_PATH = DATA_DIR / "metadata_with_embeddings_3000.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Embeddings exists:", EMBEDDINGS_PATH.exists())
print("Metadata exists:", METADATA_PATH.exists())

## 1. Install and import dependencies

In [ ]:
!pip install -q open_clip_torch

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, TensorDataset
import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Load and verify Part 1 outputs

In [ ]:
embeddings = np.load(EMBEDDINGS_PATH).astype("float32")
metadata = pd.read_csv(METADATA_PATH)

print("Embeddings:", embeddings.shape)
print("Metadata:", metadata.shape)
print("Metadata columns:", metadata.columns.tolist())
display(metadata.head())

assert embeddings.ndim == 2
assert embeddings.shape[1] == 512
assert len(metadata) == embeddings.shape[0]
assert "image_id" in metadata.columns

embedding_norms = np.linalg.norm(embeddings, axis=1)
print("Embedding norm mean:", embedding_norms.mean())
print("Embedding norm std:", embedding_norms.std())
print("Min norm:", embedding_norms.min())
print("Max norm:", embedding_norms.max())

## 3. Train Sparse Autoencoder

In [ ]:
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=1024):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        return F.relu(self.encoder(x))

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

In [ ]:
INPUT_DIM = 512
HIDDEN_DIM = 1024
EPOCHS = 100
BATCH_SIZE = 128
LR = 1e-3
L1_LAMBDA = 5e-4

x = torch.tensor(embeddings, dtype=torch.float32)
dataset = TensorDataset(x)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = SparseAutoencoder(INPUT_DIM, HIDDEN_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = []
print(model)

In [ ]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_recon = 0.0
    total_l1 = 0.0

    for (xb,) in loader:
        xb = xb.to(device)
        optimizer.zero_grad()

        x_hat, z = model(xb)

        recon_loss = F.mse_loss(x_hat, xb)
        l1_loss = z.abs().mean()
        loss = recon_loss + L1_LAMBDA * l1_loss

        loss.backward()
        optimizer.step()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        total_recon += recon_loss.item() * bs
        total_l1 += l1_loss.item() * bs

    n = len(dataset)
    row = {
        "epoch": epoch,
        "loss": total_loss / n,
        "reconstruction_loss": total_recon / n,
        "l1_loss": total_l1 / n,
    }
    history.append(row)

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"loss={row['loss']:.6f} | "
            f"recon={row['reconstruction_loss']:.6f} | "
            f"l1={row['l1_loss']:.6f}"
        )

In [ ]:
SAE_MODEL_PATH = OUTPUT_DIR / "sae_model.pt"
LOSS_CSV_PATH = OUTPUT_DIR / "training_loss.csv"

torch.save({
    "model_state_dict": model.state_dict(),
    "input_dim": INPUT_DIM,
    "hidden_dim": HIDDEN_DIM,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "l1_lambda": L1_LAMBDA,
}, SAE_MODEL_PATH)

loss_df = pd.DataFrame(history)
loss_df.to_csv(LOSS_CSV_PATH, index=False)

print("Saved:", SAE_MODEL_PATH)
print("Saved:", LOSS_CSV_PATH)
display(loss_df.tail())

plt.figure()
plt.plot(loss_df["epoch"], loss_df["loss"], label="total loss")
plt.plot(loss_df["epoch"], loss_df["reconstruction_loss"], label="reconstruction")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Sparse Autoencoder Training Loss")
plt.legend()
plt.show()

## 4. Compute SAE activations

In [ ]:
model.eval()
all_activations = []

activation_loader = DataLoader(dataset, batch_size=256, shuffle=False)

with torch.no_grad():
    for (xb,) in activation_loader:
        xb = xb.to(device)
        z = model.encode(xb)
        all_activations.append(z.cpu().numpy())

activations = np.concatenate(all_activations, axis=0).astype("float32")

ACTIVATIONS_PATH = OUTPUT_DIR / "sae_activations.npy"
np.save(ACTIVATIONS_PATH, activations)

print("Saved:", ACTIVATIONS_PATH)
print("Activations:", activations.shape)

## 5. Activation diagnostics and neuron filtering

In [ ]:
mean_activation = activations.mean(axis=0)
max_activation = activations.max(axis=0)
active_fraction = (activations > 1e-6).mean(axis=0)

dead_neuron_mask = max_activation <= 1e-6

MIN_ACTIVE_FRACTION = 0.005
MAX_ACTIVE_FRACTION = 0.98

too_rare_mask = active_fraction < MIN_ACTIVE_FRACTION
always_on_mask = active_fraction > MAX_ACTIVE_FRACTION

usable_neuron_mask = (
    (~dead_neuron_mask)
    & (~too_rare_mask)
    & (~always_on_mask)
)

print("Total neurons:", activations.shape[1])
print("Dead neurons:", int(dead_neuron_mask.sum()))
print("Too rare neurons:", int(too_rare_mask.sum()))
print("Always-on neurons:", int(always_on_mask.sum()))
print("Usable neurons after filtering:", int(usable_neuron_mask.sum()))
print("Overall activation sparsity:", float((activations <= 1e-6).mean()))

activation_stats = pd.DataFrame({
    "neuron": np.arange(activations.shape[1]),
    "mean_activation": mean_activation,
    "max_activation": max_activation,
    "active_fraction": active_fraction,
    "is_dead": dead_neuron_mask,
    "is_too_rare": too_rare_mask,
    "is_always_on": always_on_mask,
    "is_usable": usable_neuron_mask,
})

activation_stats.to_csv(OUTPUT_DIR / "activation_stats.csv", index=False)

display(activation_stats.sort_values("mean_activation", ascending=False).head(30))

plt.figure()
plt.hist(active_fraction, bins=60)
plt.title("Active fraction per neuron")
plt.xlabel("Fraction of images where neuron is active")
plt.ylabel("Number of neurons")
plt.show()

## 6. Concept vocabulary

In [ ]:
CONCEPTS = [
    "normal chest x-ray",
    "clear lungs",
    "no acute cardiopulmonary abnormality",
    "no focal consolidation",
    "normal heart size",

    "cardiomegaly",
    "enlarged cardiac silhouette",
    "enlarged heart",
    "mild cardiomegaly",
    "severe cardiomegaly",
    "widened mediastinum",

    "pleural effusion",
    "small pleural effusion",
    "large pleural effusion",
    "left pleural effusion",
    "right pleural effusion",
    "bilateral pleural effusions",
    "pleural thickening",

    "lung opacity",
    "focal lung opacity",
    "left lung opacity",
    "right lung opacity",
    "bilateral lung opacities",
    "consolidation",
    "left lower lobe consolidation",
    "right lower lobe consolidation",
    "pneumonia",
    "airspace disease",
    "infiltrate",
    "bilateral infiltrates",

    "atelectasis",
    "left basilar atelectasis",
    "right basilar atelectasis",
    "bibasilar atelectasis",
    "low lung volumes",

    "pulmonary edema",
    "interstitial pulmonary edema",
    "vascular congestion",
    "pulmonary vascular congestion",
    "interstitial markings",
    "diffuse interstitial opacities",

    "pneumothorax",
    "small pneumothorax",
    "left pneumothorax",
    "right pneumothorax",
    "no pneumothorax",

    "support devices",
    "pacemaker",
    "cardiac pacemaker",
    "central venous catheter",
    "endotracheal tube",
    "nasogastric tube",
    "chest tube",
    "medical device",

    "rib fracture",
    "fracture",
    "post surgical changes",
    "sternotomy wires",
    "calcification",

    "hyperinflation",
    "emphysema",
    "lung lesion",
    "pulmonary nodule",
    "mass",
]

TEXT_TEMPLATES = [
    "{}",
    "a chest x-ray showing {}",
    "a radiograph with {}",
    "frontal chest x-ray with {}",
]

## 7. Encode text concepts with CLIP

In [ ]:
clip_model, _, _ = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai",
)

tokenizer = open_clip.get_tokenizer("ViT-B-32")

clip_model = clip_model.to(device)
clip_model.eval()


def encode_concepts_with_templates(concepts, templates):
    concept_embeddings = []

    with torch.no_grad():
        for concept in concepts:
            prompt_embeddings = []

            for template in templates:
                text = template.format(concept)
                tokens = tokenizer([text]).to(device)

                emb = clip_model.encode_text(tokens)
                emb = F.normalize(emb, dim=-1)

                prompt_embeddings.append(emb)

            prompt_embeddings = torch.cat(prompt_embeddings, dim=0)
            concept_emb = prompt_embeddings.mean(dim=0, keepdim=True)
            concept_emb = F.normalize(concept_emb, dim=-1)

            concept_embeddings.append(concept_emb.cpu())

    return torch.cat(concept_embeddings, dim=0).numpy().astype("float32")


concept_embeddings = encode_concepts_with_templates(CONCEPTS, TEXT_TEMPLATES)
print("Concept embeddings:", concept_embeddings.shape)

## 8. Build neuron prototypes

In [ ]:
def normalize_np(x, axis=-1, eps=1e-8):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)


def build_neuron_prototypes(embeddings, activations, usable_mask, top_k=50):
    n_neurons = activations.shape[1]
    embedding_dim = embeddings.shape[1]

    prototypes = np.zeros((n_neurons, embedding_dim), dtype="float32")

    for neuron_idx in range(n_neurons):
        if not usable_mask[neuron_idx]:
            continue

        scores = activations[:, neuron_idx]
        top_idx = np.argsort(scores)[-top_k:]
        top_scores = scores[top_idx]
        top_embeddings = embeddings[top_idx]

        if top_scores.sum() <= 1e-8:
            continue

        weights = top_scores / (top_scores.sum() + 1e-8)
        prototype = np.sum(top_embeddings * weights[:, None], axis=0)

        prototypes[neuron_idx] = prototype

    return normalize_np(prototypes)


TOP_K_IMAGES_FOR_NEURON = 50

neuron_prototypes = build_neuron_prototypes(
    embeddings=embeddings,
    activations=activations,
    usable_mask=usable_neuron_mask,
    top_k=TOP_K_IMAGES_FOR_NEURON,
)

print("Neuron prototypes:", neuron_prototypes.shape)

## 9. Assign neuron concepts

In [ ]:
SIMILARITY_THRESHOLD = 0.20
MARGIN_THRESHOLD = 0.002

similarities = neuron_prototypes @ concept_embeddings.T

neuron_to_concept = {}
rows = []

for neuron_idx in range(similarities.shape[0]):

    if dead_neuron_mask[neuron_idx]:
        status = "dead"
        assigned_concept = "dead_neuron"
        best_score = 0.0
        margin = 0.0
        top_candidates = []

    elif too_rare_mask[neuron_idx]:
        status = "too_rare"
        assigned_concept = "filtered_too_rare"
        best_score = 0.0
        margin = 0.0
        top_candidates = []

    elif always_on_mask[neuron_idx]:
        status = "always_on"
        assigned_concept = "filtered_always_on"
        best_score = 0.0
        margin = 0.0
        top_candidates = []

    else:
        sim_row = similarities[neuron_idx]
        sorted_idx = np.argsort(sim_row)[::-1]

        best_idx = int(sorted_idx[0])
        second_idx = int(sorted_idx[1])

        best_score = float(sim_row[best_idx])
        second_score = float(sim_row[second_idx])
        margin = best_score - second_score

        top5_idx = sorted_idx[:5]
        top_candidates = [
            {"concept": CONCEPTS[int(i)], "score": float(sim_row[int(i)])}
            for i in top5_idx
        ]

        if best_score < SIMILARITY_THRESHOLD:
            assigned_concept = "unassigned"
            status = "low_similarity"
        elif margin < MARGIN_THRESHOLD:
            assigned_concept = "ambiguous"
            status = "ambiguous"
        else:
            assigned_concept = CONCEPTS[best_idx]
            status = "assigned"

    neuron_to_concept[str(neuron_idx)] = {
        "concept": assigned_concept,
        "score": float(best_score),
        "margin": float(margin),
        "status": status,
        "top_candidates": top_candidates,
        "mean_activation": float(mean_activation[neuron_idx]),
        "max_activation": float(max_activation[neuron_idx]),
        "active_fraction": float(active_fraction[neuron_idx]),
    }

    rows.append({
        "neuron": neuron_idx,
        "concept": assigned_concept,
        "score": float(best_score),
        "margin": float(margin),
        "status": status,
        "mean_activation": float(mean_activation[neuron_idx]),
        "max_activation": float(max_activation[neuron_idx]),
        "active_fraction": float(active_fraction[neuron_idx]),
        "top_candidates": "; ".join(
            [f"{x['concept']}={x['score']:.3f}" for x in top_candidates]
        ),
    })

mapping_df = pd.DataFrame(rows)

with open(OUTPUT_DIR / "neuron_to_concept.json", "w") as f:
    json.dump(neuron_to_concept, f, indent=2)

mapping_df.to_csv(OUTPUT_DIR / "neuron_to_concept.csv", index=False)

print("Status counts:")
display(mapping_df["status"].value_counts())

print("Assigned concept counts:")
display(mapping_df[mapping_df["status"] == "assigned"]["concept"].value_counts().head(30))
display(mapping_df.head(30))

## 10. Generate unfiltered image concepts

In [ ]:
TOP_K_NEURONS_RAW = 20
MAX_RAW_CONCEPTS_PER_IMAGE = 5

raw_image_concepts = {}
raw_rows = []

for image_idx in range(activations.shape[0]):
    image_id = str(metadata.loc[image_idx, "image_id"])
    scores = activations[image_idx]
    top_neurons = np.argsort(scores)[-TOP_K_NEURONS_RAW:][::-1]

    selected = []
    seen = set()

    for neuron_idx in top_neurons:
        neuron_idx = int(neuron_idx)
        info = neuron_to_concept[str(neuron_idx)]

        if info["status"] not in ["assigned", "ambiguous"]:
            continue

        if info["status"] == "ambiguous":
            if len(info["top_candidates"]) == 0:
                continue
            concept = info["top_candidates"][0]["concept"]
        else:
            concept = info["concept"]

        if concept in seen:
            continue

        seen.add(concept)

        item = {
            "concept": concept,
            "activation": float(scores[neuron_idx]),
            "neuron": neuron_idx,
            "status": info["status"],
            "naming_similarity": float(info["score"]),
            "margin": float(info["margin"]),
        }

        selected.append(item)

        raw_rows.append({
            "image_id": image_id,
            "rank": len(selected),
            "concept": concept,
            "activation": float(scores[neuron_idx]),
            "neuron": neuron_idx,
            "status": info["status"],
            "naming_similarity": float(info["score"]),
            "margin": float(info["margin"]),
        })

        if len(selected) >= MAX_RAW_CONCEPTS_PER_IMAGE:
            break

    raw_image_concepts[image_id] = selected

with open(OUTPUT_DIR / "image_concepts_unfiltered.json", "w") as f:
    json.dump(raw_image_concepts, f, indent=2)

raw_image_concepts_df = pd.DataFrame(raw_rows)
raw_image_concepts_df.to_csv(OUTPUT_DIR / "image_concepts_unfiltered.csv", index=False)

print("Raw image-concept rows:", len(raw_image_concepts_df))

if len(raw_image_concepts_df) > 0:
    print("Images with raw concepts:", raw_image_concepts_df["image_id"].nunique())
    print("Unique raw concepts:", raw_image_concepts_df["concept"].nunique())
    display(raw_image_concepts_df["concept"].value_counts().head(30))
    display(raw_image_concepts_df.head(20))

## 11. Generate filtered image concepts

In [ ]:
TOP_K_NEURONS_TO_SCAN = 100
MAX_CONCEPTS_PER_IMAGE = 5

IMAGE_Z_THRESHOLD = 1.5
MIN_ABSOLUTE_ACTIVATION = 0.01

image_concepts = {}
image_rows = []

for image_idx in range(activations.shape[0]):
    image_id = str(metadata.loc[image_idx, "image_id"])
    scores = activations[image_idx]

    image_mean = float(scores.mean())
    image_std = float(scores.std())
    threshold = image_mean + IMAGE_Z_THRESHOLD * image_std
    threshold = max(threshold, MIN_ABSOLUTE_ACTIVATION)

    top_neurons = np.argsort(scores)[-TOP_K_NEURONS_TO_SCAN:][::-1]

    selected = []
    seen_concepts = set()

    for neuron_idx in top_neurons:
        neuron_idx = int(neuron_idx)
        activation_score = float(scores[neuron_idx])

        if activation_score < threshold:
            continue

        info = neuron_to_concept[str(neuron_idx)]

        if info["status"] != "assigned":
            continue

        concept = info["concept"]

        if concept in seen_concepts:
            continue

        seen_concepts.add(concept)

        item = {
            "concept": concept,
            "activation": activation_score,
            "neuron": neuron_idx,
            "naming_similarity": float(info["score"]),
            "margin": float(info["margin"]),
            "image_threshold": float(threshold),
        }

        selected.append(item)

        image_rows.append({
            "image_id": image_id,
            "rank": len(selected),
            "concept": concept,
            "activation": activation_score,
            "neuron": neuron_idx,
            "naming_similarity": float(info["score"]),
            "margin": float(info["margin"]),
            "image_threshold": float(threshold),
        })

        if len(selected) >= MAX_CONCEPTS_PER_IMAGE:
            break

    image_concepts[image_id] = selected

with open(OUTPUT_DIR / "image_concepts_filtered.json", "w") as f:
    json.dump(image_concepts, f, indent=2)

image_concepts_df = pd.DataFrame(image_rows)
image_concepts_df.to_csv(OUTPUT_DIR / "image_concepts_filtered.csv", index=False)

print("Filtered image-concept rows:", len(image_concepts_df))

if len(image_concepts_df) > 0:
    print("Images with at least one filtered concept:", image_concepts_df["image_id"].nunique())
    print("Unique filtered concepts:", image_concepts_df["concept"].nunique())
    display(image_concepts_df["concept"].value_counts().head(30))
    display(image_concepts_df.head(30))
else:
    print("No filtered image concepts were produced.")

## 12. Evaluation 1 - Over-activation / generalness

In [ ]:
def compute_concept_generalness(df, num_images, concept_column="concept"):
    if len(df) == 0:
        return pd.DataFrame(columns=["concept", "image_count", "image_fraction", "generalness_label"])

    counts = df.groupby(concept_column)["image_id"].nunique().reset_index(name="image_count")
    counts["image_fraction"] = counts["image_count"] / num_images

    def label_generalness(frac):
        if frac > 0.80:
            return "over_activated"
        elif frac > 0.30:
            return "broad"
        else:
            return "specific"

    counts["generalness_label"] = counts["image_fraction"].apply(label_generalness)
    return counts.sort_values("image_fraction", ascending=False)


raw_generalness_df = compute_concept_generalness(raw_image_concepts_df, len(metadata))
filtered_generalness_df = compute_concept_generalness(image_concepts_df, len(metadata))

raw_generalness_df.to_csv(OUTPUT_DIR / "generalness_unfiltered.csv", index=False)
filtered_generalness_df.to_csv(OUTPUT_DIR / "generalness_filtered.csv", index=False)

print("Unfiltered concept generalness:")
display(raw_generalness_df)

print("Filtered concept generalness:")
display(filtered_generalness_df)

## 13. Evaluation 2 - Semantic alignment

In [ ]:
assigned_df = mapping_df[mapping_df["status"] == "assigned"].copy()
ambiguous_df = mapping_df[mapping_df["status"] == "ambiguous"].copy()

alignment_summary = {
    "assigned_neurons": int(len(assigned_df)),
    "ambiguous_neurons": int(len(ambiguous_df)),
    "mean_assigned_similarity": float(assigned_df["score"].mean()) if len(assigned_df) else 0.0,
    "median_assigned_similarity": float(assigned_df["score"].median()) if len(assigned_df) else 0.0,
    "mean_assigned_margin": float(assigned_df["margin"].mean()) if len(assigned_df) else 0.0,
    "median_assigned_margin": float(assigned_df["margin"].median()) if len(assigned_df) else 0.0,
    "mean_ambiguous_similarity": float(ambiguous_df["score"].mean()) if len(ambiguous_df) else 0.0,
    "mean_ambiguous_margin": float(ambiguous_df["margin"].mean()) if len(ambiguous_df) else 0.0,
}

alignment_summary_df = pd.DataFrame([alignment_summary])
alignment_summary_df.to_csv(OUTPUT_DIR / "semantic_alignment_summary.csv", index=False)

display(alignment_summary_df)

if len(assigned_df) > 0:
    plt.figure()
    plt.hist(assigned_df["score"], bins=30)
    plt.title("Assigned neuron semantic similarity")
    plt.xlabel("Cosine similarity")
    plt.ylabel("Number of neurons")
    plt.show()

    plt.figure()
    plt.hist(assigned_df["margin"], bins=30)
    plt.title("Assigned neuron top1-top2 margin")
    plt.xlabel("Margin")
    plt.ylabel("Number of neurons")
    plt.show()

## 14. Evaluation 3 - Before/after filtering

In [ ]:
before_after_summary = {
    "raw_rows": int(len(raw_image_concepts_df)),
    "filtered_rows": int(len(image_concepts_df)),
    "raw_images_with_concepts": int(raw_image_concepts_df["image_id"].nunique()) if len(raw_image_concepts_df) else 0,
    "filtered_images_with_concepts": int(image_concepts_df["image_id"].nunique()) if len(image_concepts_df) else 0,
    "raw_unique_concepts": int(raw_image_concepts_df["concept"].nunique()) if len(raw_image_concepts_df) else 0,
    "filtered_unique_concepts": int(image_concepts_df["concept"].nunique()) if len(image_concepts_df) else 0,
}

before_after_summary["raw_mean_concepts_per_image"] = (
    float(raw_image_concepts_df.groupby("image_id").size().mean())
    if len(raw_image_concepts_df) else 0.0
)

before_after_summary["filtered_mean_concepts_per_image"] = (
    float(image_concepts_df.groupby("image_id").size().mean())
    if len(image_concepts_df) else 0.0
)

before_after_df = pd.DataFrame([before_after_summary])
before_after_df.to_csv(OUTPUT_DIR / "before_after_filtering_summary.csv", index=False)

display(before_after_df)

print("Top raw concepts:")
if len(raw_image_concepts_df) > 0:
    display(raw_image_concepts_df["concept"].value_counts().head(20))

print("Top filtered concepts:")
if len(image_concepts_df) > 0:
    display(image_concepts_df["concept"].value_counts().head(20))

## 15. Evaluation 4 - Similar-image concept consistency

In [ ]:
def get_concept_sets_from_df(df):
    concept_sets = {str(image_id): set() for image_id in metadata["image_id"].astype(str).tolist()}

    if len(df) == 0:
        return concept_sets

    grouped = df.groupby("image_id")["concept"].apply(set)

    for image_id, concepts in grouped.items():
        concept_sets[str(image_id)] = set(concepts)

    return concept_sets


def jaccard(set_a, set_b):
    if len(set_a) == 0 and len(set_b) == 0:
        return 1.0

    union = set_a | set_b

    if len(union) == 0:
        return 0.0

    return len(set_a & set_b) / len(union)


def compute_nearest_neighbors(embeddings, k=5):
    emb = embeddings.astype("float32")
    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-8)

    sim = emb @ emb.T
    np.fill_diagonal(sim, -np.inf)

    nn_idx = np.argsort(sim, axis=1)[:, -k:][:, ::-1]
    nn_sim = np.take_along_axis(sim, nn_idx, axis=1)

    return nn_idx, nn_sim


K_NEIGHBORS = 5

nn_idx, nn_sim = compute_nearest_neighbors(embeddings, k=K_NEIGHBORS)

filtered_concept_sets = get_concept_sets_from_df(image_concepts_df)
raw_concept_sets = get_concept_sets_from_df(raw_image_concepts_df)

image_ids = metadata["image_id"].astype(str).tolist()

rows = []

for i, image_id in enumerate(image_ids):
    source_filtered = filtered_concept_sets[image_id]
    source_raw = raw_concept_sets[image_id]

    for rank in range(K_NEIGHBORS):
        neighbor_index = int(nn_idx[i, rank])
        neighbor_id = image_ids[neighbor_index]

        neighbor_filtered = filtered_concept_sets[neighbor_id]
        neighbor_raw = raw_concept_sets[neighbor_id]

        rows.append({
            "image_id": image_id,
            "neighbor_rank": rank + 1,
            "neighbor_image_id": neighbor_id,
            "embedding_similarity": float(nn_sim[i, rank]),
            "filtered_jaccard": float(jaccard(source_filtered, neighbor_filtered)),
            "raw_jaccard": float(jaccard(source_raw, neighbor_raw)),
            "source_filtered_concepts": "; ".join(sorted(source_filtered)),
            "neighbor_filtered_concepts": "; ".join(sorted(neighbor_filtered)),
            "source_raw_concepts": "; ".join(sorted(source_raw)),
            "neighbor_raw_concepts": "; ".join(sorted(neighbor_raw)),
        })

consistency_pairs_df = pd.DataFrame(rows)
consistency_pairs_df.to_csv(OUTPUT_DIR / "similar_image_consistency_pairs.csv", index=False)

consistency_summary = {
    "k_neighbors": K_NEIGHBORS,
    "mean_filtered_jaccard": float(consistency_pairs_df["filtered_jaccard"].mean()),
    "median_filtered_jaccard": float(consistency_pairs_df["filtered_jaccard"].median()),
    "mean_raw_jaccard": float(consistency_pairs_df["raw_jaccard"].mean()),
    "median_raw_jaccard": float(consistency_pairs_df["raw_jaccard"].median()),
}

consistency_summary_df = pd.DataFrame([consistency_summary])
consistency_summary_df.to_csv(OUTPUT_DIR / "similar_image_consistency_summary.csv", index=False)

print("Similar-image consistency summary:")
display(consistency_summary_df)
display(consistency_pairs_df.head(20))

plt.figure()
plt.hist(consistency_pairs_df["filtered_jaccard"], bins=30)
plt.title("Filtered concept consistency across nearest neighbors")
plt.xlabel("Jaccard similarity")
plt.ylabel("Number of image-neighbor pairs")
plt.show()

## 16. Evaluation 5 - Concept-level consistency score

In [ ]:
def compute_concept_level_consistency(concept_sets, concept_list, nn_idx, image_ids):
    rows = []

    for concept in concept_list:
        image_indices_with_concept = [
            i for i, image_id in enumerate(image_ids)
            if concept in concept_sets[image_id]
        ]

        if len(image_indices_with_concept) == 0:
            rows.append({
                "concept": concept,
                "image_count": 0,
                "neighbor_checks": 0,
                "same_concept_neighbor_count": 0,
                "concept_consistency": 0.0,
            })
            continue

        checks = 0
        same = 0

        for i in image_indices_with_concept:
            for neighbor_index in nn_idx[i]:
                neighbor_id = image_ids[int(neighbor_index)]
                checks += 1

                if concept in concept_sets[neighbor_id]:
                    same += 1

        consistency = same / checks if checks > 0 else 0.0

        rows.append({
            "concept": concept,
            "image_count": len(image_indices_with_concept),
            "neighbor_checks": checks,
            "same_concept_neighbor_count": same,
            "concept_consistency": consistency,
        })

    return pd.DataFrame(rows).sort_values("concept_consistency", ascending=False)


filtered_concepts = (
    sorted(image_concepts_df["concept"].unique().tolist())
    if len(image_concepts_df) > 0 else []
)

raw_concepts = (
    sorted(raw_image_concepts_df["concept"].unique().tolist())
    if len(raw_image_concepts_df) > 0 else []
)

filtered_concept_consistency_df = compute_concept_level_consistency(
    filtered_concept_sets,
    filtered_concepts,
    nn_idx,
    image_ids,
)

raw_concept_consistency_df = compute_concept_level_consistency(
    raw_concept_sets,
    raw_concepts,
    nn_idx,
    image_ids,
)

filtered_concept_consistency_df.to_csv(
    OUTPUT_DIR / "concept_consistency_filtered.csv",
    index=False,
)

raw_concept_consistency_df.to_csv(
    OUTPUT_DIR / "concept_consistency_unfiltered.csv",
    index=False,
)

print("Filtered concept-level consistency:")
display(filtered_concept_consistency_df)

print("Unfiltered concept-level consistency:")
display(raw_concept_consistency_df.head(30))

## 17. Final reliability report

In [ ]:
report = {}

report["pipeline"] = {
    "num_images": int(embeddings.shape[0]),
    "embedding_dim": int(embeddings.shape[1]),
    "num_sae_neurons": int(activations.shape[1]),
    "dead_neurons": int(dead_neuron_mask.sum()),
    "too_rare_neurons": int(too_rare_mask.sum()),
    "always_on_neurons": int(always_on_mask.sum()),
    "usable_neurons": int(usable_neuron_mask.sum()),
    "assigned_neurons": int(len(assigned_df)),
    "ambiguous_neurons": int(len(ambiguous_df)),
}

report["image_concepts"] = before_after_summary
report["semantic_alignment"] = alignment_summary
report["similar_image_consistency"] = consistency_summary

report["filtered_generalness"] = (
    filtered_generalness_df.to_dict(orient="records")
    if len(filtered_generalness_df) > 0 else []
)

report["filtered_concept_consistency"] = (
    filtered_concept_consistency_df.to_dict(orient="records")
    if len(filtered_concept_consistency_df) > 0 else []
)

REPORT_JSON_PATH = OUTPUT_DIR / "reliability_report.json"

with open(REPORT_JSON_PATH, "w") as f:
    json.dump(report, f, indent=2)

print("Saved:", REPORT_JSON_PATH)

print(json.dumps(report["pipeline"], indent=2))
print(json.dumps(report["image_concepts"], indent=2))
print(json.dumps(report["semantic_alignment"], indent=2))
print(json.dumps(report["similar_image_consistency"], indent=2))

## 18. Final outputs

In [ ]:
print("FINAL OUTPUT FILES:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name)

print("\nMain deliverables:")
print("- sae_model.pt")
print("- sae_activations.npy")
print("- neuron_to_concept.json / .csv")
print("- image_concepts_filtered.json / .csv")
print("- reliability_report.json")
print("- semantic_alignment_summary.csv")
print("- before_after_filtering_summary.csv")
print("- similar_image_consistency_summary.csv")
print("- concept_consistency_filtered.csv")
print("- generalness_filtered.csv")